# L88 · TF-IDF 检索从零实现 — 词频逆文档频率，纯 NumPy 向量检索

**学习目标**
- 理解 TF-IDF 稀疏表示（Sparse Representation）的原理
- 从零实现 build_tfidf() 和 cosine_retrieve()
- 与 aurora.llm.retrieve 对比验证
- 用它检索一个小型 DSP 知识库

## 为什么用 TF-IDF 而不是 sentence embeddings？

**Sentence embeddings**（SentenceTransformer）：需要预训练模型 (~400MB)，离不开 GPU，黑盒。

**TF-IDF**：纯数学，不需要任何预训练权重，可完全从零理解，在关键词匹配任务上往往已经足够好。

### TF-IDF 公式
```
TF(t,d)  = 词 t 在文档 d 中的出现次数 / 文档 d 的总词数
IDF(t)   = log( (N+1) / (df(t)+1) ) + 1       （smoothed，df=含 t 的文档数）
TF-IDF   = TF × IDF
```

**查询向量**：用同样的 tokenize + TF 计算（**不乘 IDF**），然后求余弦相似度（Cosine Similarity）。

> **为什么查询不乘 IDF？** IDF 对高频词的惩罚在文档端已经完成；对查询再次施加会过度抑制常见关键词，实验表明不乘效果更好（BM25 同样采用此设计）。

In [ ]:
import numpy as np
import re
from collections import Counter

In [ ]:
# 小型 DSP / 音频 AI 知识库（10 个文档）
DOCS = [
    'The Fourier transform decomposes a signal into its frequency components',
    'FFT stands for Fast Fourier Transform and runs in O N log N time',
    'The STFT applies a windowed Fourier transform to produce a spectrogram',
    'Mel filterbanks simulate human auditory perception of frequency',
    'MFCC Mel Frequency Cepstral Coefficients are computed via DCT after mel filterbank',
    'The Nyquist theorem states the sampling rate must be at least twice the maximum frequency',
    'Window functions like Hann and Hamming reduce spectral leakage in FFT',
    'CTC Connectionist Temporal Classification allows sequence labeling without alignment',
    'Whisper is a Transformer based speech recognition model from OpenAI',
    'Beat tracking estimates the tempo of music using onset detection and autocorrelation',
]
print(f'知识库：{len(DOCS)} 个文档')

## ✏️ 任务 1：实现 tokenize

In [ ]:
def tokenize(text: str) -> list:
    """简单 tokenizer：提取小写 ASCII 单词。"""
    # ✏️ TODO: 用正则提取单词，转小写
    raise NotImplementedError("TODO")

assert tokenize('Hello World!') == ['hello', 'world']
assert tokenize('FFT O(N log N)') == ['fft', 'o', 'n', 'log', 'n']
print('✅ tokenize 验证通过')

## ✏️ 任务 2：实现 build_tfidf

In [ ]:
def build_tfidf(docs: list) -> tuple:
    """构建 TF-IDF 矩阵。返回 (matrix, vocab)。"""
    tokenized = [tokenize(d) for d in docs]
    all_terms = sorted({t for tokens in tokenized for t in tokens})
    word_idx = {w: i for i, w in enumerate(all_terms)}

    # ✏️ TODO:
    # (1) 计算 TF 矩阵 (n_docs, vocab_size)：TF(t,d) = count(t,d) / |d|
    # (2) 计算 IDF 向量 (vocab_size,)：IDF(t) = log((N+1)/(df(t)+1)) + 1
    #     df(t) = 含词 t 的文档数，N = 文档总数
    # (3) 返回 (TF × IDF, all_terms) — TF-IDF 矩阵和词汇表列表
    raise NotImplementedError("请实现 build_tfidf")

try:
    matrix, vocab = build_tfidf(DOCS)
    print(f'TF-IDF 矩阵 shape: {matrix.shape}  (n_docs={len(DOCS)}, vocab={len(vocab)})')
    print(f'词汇表前 10 词: {vocab[:10]}')
except NotImplementedError:
    matrix = None
    vocab = []
    print("⚠️  build_tfidf 尚未实现，matrix/vocab 已设为占位值。完成 TODO 后重新运行本格。")


## ✏️ 任务 3：实现 cosine_retrieve

**余弦相似度公式**
$$\cos(q,\,d) = \frac{q \cdot d}{\|q\|\,\|d\|}$$

- $q$ — 查询向量（TF，不乘 IDF）
- $d$ — 文档行向量（TF-IDF）
- 分母为零时（空查询或全零文档）得分记为 0

实现步骤：
1. 用查询词的 TF 构造 $q$（词不在词表中直接忽略）
2. 计算点积 $\text{scores}[i] = \text{tfidf\_matrix}[i] \cdot q$
3. 除以各自 L2 范数（零向量得 0 分）
4. 取 top-k 并按得分降序排列，返回 `[(doc, score), ...]`


In [ ]:
def cosine_retrieve(query: str, tfidf_matrix, vocab: list, docs: list, top_k=3) -> list:
    """余弦相似度检索，返回 (doc, score) 列表。"""
    word_idx = {w: i for i, w in enumerate(vocab)}
    tokens = tokenize(query)
    counts = Counter(tokens)
    total = max(len(tokens), 1)

    # ✏️ TODO:
    # (1) 构建查询向量 q_vec (vocab_size,)：q_vec[word_idx[t]] = counts[t] / total
    #     词不在 word_idx 中时忽略
    # (2) 计算点积：scores = tfidf_matrix @ q_vec  → shape (n_docs,)
    # (3) 余弦归一化：score_i /= (‖tfidf_matrix[i]‖ × ‖q_vec‖)
    #     若 ‖q_vec‖ == 0 或 ‖doc_vec‖ == 0，该文档得分置为 0
    # (4) 取 top_k：按得分降序排列，返回 [(docs[i], float(scores[i])), ...]
    raise NotImplementedError("请实现 cosine_retrieve")

# 测试（需先完成 build_tfidf 和 cosine_retrieve）
if matrix is not None:
    try:
        print('\n查询: "Fourier transform frequency"')
        for doc, score in cosine_retrieve('Fourier transform frequency', matrix, vocab, DOCS):
            print(f'  [{score:.3f}] {doc[:60]}')
        print('\n查询: "speech recognition neural network"')
        for doc, score in cosine_retrieve('speech recognition neural network', matrix, vocab, DOCS):
            print(f'  [{score:.3f}] {doc[:60]}')
    except NotImplementedError:
        print("⚠️  cosine_retrieve 尚未实现，完成 TODO 后重新运行本格。")
else:
    print("⚠️  matrix 为 None，请先完成 build_tfidf。")


## Aurora 连接

本课的两个函数对应 `src/aurora/llm/retrieve.py`：

| 本课实现 | 生产代码 |
|---|---|
| `build_tfidf(docs)` | `aurora.llm.build_tfidf` |
| `cosine_retrieve(query, ...)` | `aurora.llm.cosine_retrieve` |

下一课（L89）将直接 `from aurora.llm import ...` 把这个检索器接入完整的 RAG 流水线。


In [ ]:
# 与 aurora.llm 对比验证（需先完成两个 TODO）
from aurora.llm import build_tfidf as ref_build, cosine_retrieve as ref_retrieve

ref_mat, ref_vocab = ref_build(DOCS)
print(f'aurora.llm matrix shape: {ref_mat.shape}  ← 应与自实现相同')

if matrix is not None:
    # 1. 矩阵数值对比
    assert np.allclose(matrix, ref_mat, atol=1e-5), (
        f"❌ 矩阵与参考实现不一致！\n"
        f"  自实现 shape={matrix.shape}, max={matrix.max():.4f}\n"
        f"  参考   shape={ref_mat.shape}, max={ref_mat.max():.4f}\n"
        "检查你的 TF 公式和 IDF 公式是否与 cell 1 一致。"
    )
    print("✅ build_tfidf 矩阵与参考实现一致")

    # 2. 检索顺序对比
    try:
        my_results = cosine_retrieve('Fourier transform frequency', matrix, vocab, DOCS)
        ref_results = ref_retrieve('Fourier transform frequency', ref_mat, ref_vocab, DOCS)
        my_docs = [d for d, _ in my_results]
        ref_docs = [d for d, _ in ref_results]
        assert my_docs == ref_docs, (
            f"❌ 检索顺序与参考实现不一致！\n"
            f"  自实现: {my_docs}\n"
            f"  参考:   {ref_docs}"
        )
        print("✅ cosine_retrieve 结果顺序与参考实现一致")
        print("✅ TF-IDF 检索验证通过")
    except NotImplementedError:
        print("⚠️  cosine_retrieve 尚未实现，跳过检索顺序对比。")
else:
    # 仅显示参考结果供参考
    ref_results = ref_retrieve('Fourier transform frequency', ref_mat, ref_vocab, DOCS)
    print("aurora.llm 检索结果（参考）:", [s for _, s in ref_results])
    print("⚠️  build_tfidf 尚未实现，跳过自实现对比。")


## 本课收束

| 概念 | 要记住的 |
|---|---|
| TF | 词在文档中的相对频率 |
| IDF | 含这个词的文档越多，权重越低（常见词权重低）|
| 余弦相似度 | 方向相同 → 相似，与向量长度无关 |
| 优点 | 无需预训练，可完全理解，快速 O(n×d) |
| 缺点 | 语义匹配弱（"car"≠"automobile"），无上下文理解 |
| L89 | 把这个检索器组装进完整的 RAG 流水线 |

下一课（L89）把 TF-IDF 检索、向量相似度和 LLM 生成组合成完整的 RAG 管道（RAG pipeline），端到端回答问题。